# เฟส 2 — nowcast ด้วย extrapolationตรรกะจริงอยู่ใน `radar_archive/nowcast.py` notebook นี้เป็นตัวเรียกใช้และตรวจผล**สิ่งสำคัญที่สุดที่ต้องทำใน notebook นี้:** ข้อ 4 — เทียบ engine เบาของเรากับ pystepsถ้าตรงกัน ระบบจริงใช้ engine เบาได้อย่างสบายใจ (pysteps ไม่มี wheel ต้อง compile 2-3 นาทีทุกครั้งรับความเสี่ยงนั้นในงานที่รันทุก 15 นาทีไม่ไหว)

In [ ]:
!git clone -q https://github.com/jamorn12/tmd-radar-archive.git%cd tmd-radar-archive!pip install -q -r requirements.txt!apt-get install -y -qq tesseract-ocr > /dev/nullprint("พร้อม")

## 1. รัน nowcast ด้วย engine เบา (ไม่ต้องมี pysteps)

In [ ]:
!python -m radar_archive.nowcast --station PHS --engine light --out /content/out

## 2. ดู latest.json ที่จะเสิร์ฟให้แอป

In [ ]:
import jsond = json.load(open('/content/out/latest.json'))for k in ['station','base_time_local','age_min','stale','projection','grid',          'kmperpixel','timestep_min','wet_threshold_dbz','zr']:    print(f"  {k:<20}{d[k]}")print(f"\n  motion: {json.dumps(d['motion'], ensure_ascii=False)}")print(f"  qc    : {d['qc']}")print(f"\n  {'kind':<9}{'offset':>7}{'max dBZ':>9}{'wet %':>8}")for f in d['frames']:    print(f"  {f['kind']:<9}{f['offset_min']:>+7}{str(f['max_dbz']):>9}{f['wet_pct']:>8}")

## 3. ดูภาพ — ของจริงต่อด้วยพยากรณ์แถวบนคือเฟรมจริง แถวล่างคือ extrapolation เส้นประคั่นตรงรอยต่อ

In [ ]:
import numpy as np, matplotlib.pyplot as pltfrom PIL import Imagefrom pathlib import Pathfr = d['frames']cols = len(fr)fig, axes = plt.subplots(1, cols, figsize=(2.5*cols, 3.0), facecolor='white')for ax, f in zip(np.ravel(axes), fr):    a = np.asarray(Image.open(Path('/content/out')/f['url']))    ax.imshow(np.zeros(a.shape[:2]), cmap='gray', vmin=0, vmax=1)   # พื้นดำ    ax.imshow(a)    ax.add_patch(plt.Circle((120, 120), 120, fill=False, ec='0.35', lw=.6))    fc = f['kind'] == 'nowcast'    ax.set_title(f"{'+' if f['offset_min']>=0 else ''}{f['offset_min']} min\n"                 f"{'forecast' if fc else 'observed'}", fontsize=9,                 color='#B06A18' if fc else '#166')    for s in ax.spines.values():        s.set_color('#E0A050' if fc else '#88A'); s.set_linewidth(1.6 if fc else .8)    ax.set_xticks([]); ax.set_yticks([])fig.suptitle("PHS extrapolation nowcast — observed | forecast", y=1.06)fig.tight_layout(); plt.show()

## 4. ⭐ เทียบ engine เบา กับ pystepsนี่คือขั้นที่ทำให้เราอ้างอิงได้ในเปเปอร์ว่า engine เบาให้ผลเท่ากับ pystepspysteps ไม่มี wheel บน PyPI ต้อง build จาก source — บน Colab ทำได้ (มี cython/compiler พร้อม)ใช้เวลาราว 2-3 นาที

In [ ]:
!pip install -q cython!pip install -q pysteps opencv-python-headlessimport pysteps; print("pysteps", pysteps.__version__)

In [ ]:
!python -m radar_archive.nowcast --station PHS --compare

อ่านผลอย่างไร- `agreement.mae_dbz` — ค่าเฉลี่ยที่ต่างกันต่อเซลล์ ยิ่งใกล้ 0 ยิ่งดี- `agreement.corr` — สหสัมพันธ์ระหว่างสองสนาม ควรใกล้ 1- `stability` ของแต่ละ engine — ถ้าให้ทิศกับความเร็วใกล้กัน แปลว่า motion ตรงกันด้วยถ้าไม่ตรงกัน อย่าเพิ่งเชื่อ engine เบา — ให้ดูก่อนว่าต่างเพราะอะไร(`dense_lucaskanade` ให้ motion รายพิกเซล ส่วนของเราให้เวกเตอร์เดียวทั้งภาพถ้าในเฟรมมีก้อนฝนหลายก้อนเคลื่อนคนละทิศ สองอันนี้จะต่างกันโดยธรรมชาติ)

## 5. ตรวจ despeckle — ตัวหนังสือบนแผนที่เฟรม 2026-09-03 09:15Z เคยให้ max 59.2 dBZ ซึ่งที่จริงคือคำว่า "Petchabun" บนแผนที่ที่ติดกับก้อนฝนจริง ทำให้ `drop_pale_blobs` ตัดไม่ได้

In [ ]:
from radar_archive import build_stack, nowcastfrom radar_archive.config import get_stationst = get_station("PHS")root = Path('data')sel = [f for f in build_stack.find_frames(root, "PHS")       if f[0].strftime("%m%d%H%M") in ("09030900", "09030915", "09030930")]if len(sel) == 3:    stack, times, meta, _ = build_stack.build_run(sel, st, root, verbose=False)    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), facecolor='white')    for ax, t, f in zip(axes, times, stack):        g, n = nowcast.despeckle(f)        ax.imshow(f, origin='lower', vmin=0, vmax=60, cmap='turbo')        ax.set_title(f"{t:%H:%M}Z\nbefore {np.nanmax(f):.1f} -> after {np.nanmax(g):.1f} dBZ"                     f"  ({n} cells)", fontsize=9)        ax.set_xticks([]); ax.set_yticks([])    fig.suptitle("despeckle: removes strong cells with no gradient support", y=1.04)    fig.tight_layout(); plt.show()else:    print("ไม่พบเฟรมชุดนั้นแล้ว (คลังหมุนไปแล้ว) — ข้ามได้")

## 6. ทดสอบย้อนหลัง — พยากรณ์แล้วเทียบกับสิ่งที่เกิดขึ้นจริงเลือกเวลาหนึ่งในอดีต พยากรณ์จากข้อมูลที่มี ณ ตอนนั้น แล้วเทียบกับเฟรมที่มาถึงจริงนี่คือต้นแบบของ `verify.py` ในเฟส 3

In [ ]:
from radar_archive import gridimport numpy as npframes = build_stack.find_frames(root, "PHS")runs = sorted(build_stack.split_runs(frames), key=len)run = runs[-1]                      # ช่วงที่ยาวที่สุดprint(f"ช่วงที่ใช้: {run[0][0]:%d %H:%M} - {run[-1][0]:%d %H:%M}Z  {len(run)} เฟรม")N = nowcast.N_INPUTstack, times, meta, _ = build_stack.build_run(run, st, root, verbose=False)stack, _ = nowcast.despeckle_stack(stack)THR = meta['threshold']def csi(pred, obs, thr):    ok = np.isfinite(pred) & np.isfinite(obs)    p, o = pred[ok] >= thr, obs[ok] >= thr    h, m, f = int((p&o).sum()), int((~p&o).sum()), int((p&~o).sum())    return h/(h+m+f) if (h+m+f) else np.nan, h, m, frows = []for i in range(N-1, len(times)-4):    hist = stack[i-N+1:i+1]    V, info = nowcast.estimate_motion(hist, "light", meta['kmperpixel'], meta['timestep'])    fc, _ = nowcast.run_extrapolation(hist[-1], V, "light",                                      nowcast.LEADS_MIN, meta['timestep'])    for k, lead in enumerate(nowcast.LEADS_MIN):        j = i + k + 1        if j >= len(times): break        c, *_ = csi(fc[k], stack[j], THR)        cp, *_ = csi(hist[-1], stack[j], THR)     # persistence เป็นตัวเทียบ        rows.append((lead, c, cp))import collectionsagg = collections.defaultdict(list)for lead, c, cp in rows:    if np.isfinite(c): agg[lead].append((c, cp))print(f"\n{'lead':>6}{'n':>5}{'CSI':>8}{'CSI persist':>13}{'skill':>9}")for lead in sorted(agg):    a = np.array(agg[lead])    c, cp = a[:,0].mean(), a[:,1].mean()    sk = (c-cp)/(1-cp) if cp < 1 else np.nan    print(f"{lead:>6}{len(a):>5}{c:>8.3f}{cp:>13.3f}{sk:>9.3f}")

อ่านผลอย่างไร- **CSI** ของ extrapolation ควรมากกว่า **CSI persistence** ทุก lead time  ถ้าไม่มากกว่า แปลว่าการหา motion ยังไม่ได้ช่วยอะไร- **skill** = (CSI − CSI_persist) / (1 − CSI_persist) · เป็นบวก = ดีกว่าการอยู่นิ่ง- CSI ควรลดลงตาม lead time — ถ้าไม่ลด แสดงว่ามีอะไรผิด> ตัวเลขจากไม่กี่ชั่วโมงยังสรุปอะไรไม่ได้ ต้องสะสมหลายเหตุการณ์> เฟส 3 จะให้ระบบเก็บคะแนนนี้เองทุกรอบลง CSV